In [ ]:
!pip install "audio-separator[gpu]"
!apt-get install -y ffmpeg

In [ ]:
from google.colab import drive
import shutil 
import os
import subprocess
from audio_separator.separator import Separator

drive.flush_and_unmount()
drive.mount('/content/drive')

import torch
print(torch.cuda.is_available())

Mounted at /content/drive


In [ ]:
def separate(file_path):
    separator = Separator(output_format='WAV', output_single_stem='Instrumental')
    separator.load_model()

    output_audio = separator.separate(file_path)
    return output_audio

In [ ]:
def extract_audio(input_video): # If video, extract wav
    input_audio = "input.wav"
    subprocess.run([
        "ffmpeg",
        "-y",
        "-i", input_video,
        "-vn", 
        "-acodec", "pcm_s16le",  # WAV format
        input_audio
    ], check=True)
    return input_audio

In [ ]:
def stitch_audio(input_video, output_audio, output_folder):
    file_name = os.path.splitext(os.path.basename(input_video))[0]
    output_video = os.path.join(output_folder, f"{file_name}_karaoke.mp4")
    
    subprocess.run([
    "ffmpeg",
    "-i", input_video,
    "-i", output_audio,   
    "-map", "0:v",  # video from first input
    "-map", "1:a",  # audio from second input
    "-c:v", "copy",  # copy video (no re-encoding)
    "-shortest",  # match shortest duration
    output_video
    ])

In [ ]:
!ffmpeg -i /content/drive/MyDrive/karaoke/videoplayback.mp4

In [ ]:
def main():
    input_video = "/content/drive/MyDrive/karaoke/videoplayback.mp4"
    output_folder="/content/drive/MyDrive/karaoke_output"
    input_audio = extract_audio(input_video)
    output_audio = separate(input_audio)
    print(output_audio)
    stitch_audio(input_video, output_audio[0], output_folder) # output_audio is a list
    print("Extracted:", input_audio)

main()